# Day 4: Baseline RAG with citations

Study guide: `course/week-1/day-04-baseline-rag/README.md`

Goal: question in → 5 nearest chunks → prompt → Llama → answer with citations, or "I don't know".

**Setup:** reload edited `.py` files automatically. Run once after every kernel restart.

In [2]:
%load_ext autoreload
%autoreload 2

## Step 1: Search as project code
_Filled in as we go._

In [3]:
from research_rag.retrieval.search import search

results = search("Does LoRA add inference latency?", k=5)

for result in results:
    print(f"{result.similarity:.3f}  {result.id:<24} p.{result.page_number:<3} {result.text[:80]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10697.90it/s]


0.648  2106.09685.pdf::0021     p.4   LoRA roughly converges to training the original model, while adapter-based metho...
0.586  2305.14314.pdf::0030     p.6   under what circumstances slowdowns occur from the paging process. Default LoRA h...
0.584  2106.09685.pdf::0020     p.4   a result, we simply set α to the first r we try and do not tune it. This scaling...
0.564  2106.09685.pdf::0024     p.5   350GB to 35MB)4. This allows us to train with significantly fewer GPUs and avoid...
0.546  2305.14314.pdf::0124     p.22  65B). We find LoRA r is unrelated to final performance if LoRA is used on all la...


## Step 2: The prompt
_Filled in as we go._

In [6]:
from research_rag.generation.prompts import PROMPT_VERSION, build_prompt

question = "Does LoRA add inference latency?"
results = search(question, k=5)
prompt = build_prompt(question, results)

print("Prompt version:", PROMPT_VERSION)
print(prompt)

Prompt version: v1
You are a careful research assistant.
Answer the question using ONLY the numbered context below.

Rules:
- Cite every fact with the number of the context it came from, like [1] or [2][3].
- If the context does not contain the answer, reply exactly: "I don't know based on the provided papers."
- Do not use outside knowledge. Keep the answer short.

Context:
[1] (2106.09685.pdf, page 4)
LoRA roughly converges to training the original model, while adapter-based methods converges to an MLP and prefix-based methods to a model that cannot take long input sequences. No Additional Inference Latency. When deployed in production, we can explicitly compute and store W = W0 + BA and perform inference as usual. Note that both W0 and BA are in Rd×k. When we need to switch to another downstream task, we can recover W0 by subtracting BA and then adding a different B′A′, a quick operation with very little memory overhead. Critically, this 2They represent a negligible number of parame

## Step 3: The LLM client
_Filled in as we go._

In [7]:
from research_rag.generation.llm import generate

answer = generate(prompt)    # the prompt you built in Step 2
print(answer)

No. According to [1] and [4], when deployed in production, we can explicitly compute and store W = W0 + BA and perform inference as usual, and when we need to switch to another downstream task, we can recover W0 by subtracting BA and then adding a different B′A′, a quick operation with very little memory overhead.


## Step 4: Citations
_Filled in as we go._

In [8]:
from research_rag.generation.citations import extract_citations

citations = extract_citations(answer, results)
for citation in citations:
    print(f"[{citation.number}] {citation.source}, page {citation.page_number}  ({citation.chunk_id})")

[1] 2106.09685.pdf, page 4  (2106.09685.pdf::0021)
[4] 2106.09685.pdf, page 5  (2106.09685.pdf::0024)


## Step 5: answer()
_Filled in as we go._

## Step 6: Try real questions
_Filled in as we go._

In [9]:
for question in ["What is the lost in the middle problem?", "What does ReAct combine?"]:
    print("Q:", question)
    for result in search(question, k=5):
        print(f"  {result.similarity:.3f}  {result.id:<24} p.{result.page_number:<3} {result.text[:90]}...")
    print()

Q: What is the lost in the middle problem?
  0.465  2201.11903.pdf::0222     p.43  58 - 23 = 35 balls. On Wednesday he lost 2 more so now he has 35 - 2 = 33 balls. The answe...
  0.446  2201.11903.pdf::0217     p.42  Michael started with 58 golf balls and lost 23, so he has 58 - 23 = 35. After he lost 2 mo...
  0.436  2201.11903.pdf::0187     p.35  tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The answ...
  0.428  2201.11903.pdf::0214     p.42  and there are 15 trees in the beginning, so the workers plant 21 - 15 = 6 trees. The answe...
  0.426  2201.11903.pdf::0184     p.35  . Then there were 21 trees after some more were planted. So there must have been 21 - 15 =...

Q: What does ReAct combine?
  0.462  2210.03629.pdf::0053     p.8   Act is the first demonstration of combined reasoning and action using an LLM applied to an...
  0.430  2210.03629.pdf::0039     p.6   flexibility in formulating reasoning steps, leading to more reasoning error rate tha